In [2]:
!pip install langchain langchain-community langchain-openai chromadb sentence-transformers tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.8/505.8 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
base_path = "/content/drive/MyDrive/book_recommendation/book_recommendation"

txt_path = base_path + "/tagged_description.txt"
csv_path = base_path + "/books_cleaned.csv"

In [6]:
from dotenv import load_dotenv
load_dotenv()

False

First, let's verify if a `.env` file exists in your `base_path`.

In [7]:
import os

dotenv_path = os.path.join(base_path, '.env')

if not os.path.exists(dotenv_path):
    print(f"No .env file found at: {dotenv_path}. Creating a dummy .env file.")
    # Example: Create a dummy .env file with an empty API key
    with open(dotenv_path, 'w') as f:
        f.write('OPENAI_API_KEY=your_api_key_here')
else:
    print(f"Found .env file at: {dotenv_path}")

# Now, explicitly load the .env file from the specified path
from dotenv import load_dotenv
load_dotenv(dotenv_path=dotenv_path)

# Verify if the variables are loaded (e.g., for OPENAI_API_KEY)
print(f"OPENAI_API_KEY loaded: {os.getenv('OPENAI_API_KEY') is not None}")


Found .env file at: /content/drive/MyDrive/book_recommendation/book_recommendation/.env
OPENAI_API_KEY loaded: True


In [8]:
import pandas as pd
books = pd.read_csv(csv_path)

In [9]:
books["tagged_description"].to_csv("txt_path",
index=False,
header=False)

In [10]:
raw_documents = TextLoader("txt_path",
                           encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=250, chunk_overlap=50 , separator="\n" )
documents = text_splitter.split_documents(raw_documents)

In [11]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipykernel_1847/3401734470.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [23]:
persist_directory = "/content/drive/MyDrive/book_recommendation/chroma_db"
db = Chroma.from_documents(
    documents,
    embeddings,
    persist_directory=base_path + "/chroma_db"
)
db.persist()

/tmp/ipykernel_1847/3877510792.py:7: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


In [15]:
query= "A book for children about nature"
docs= db.similarity_search(query,
                                 k=10)
docs

[Document(metadata={'source': 'txt_path'}, page_content='"9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience."'),
 Document(metadata={'source': 'txt_path'}, page_content='"9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience."'),
 Document(metadata={'source': 'txt_path'}, page_content='"9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learn

In [17]:
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip('"'))]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
3747,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,Baby Einstein: Neighborhood Animals:,9780786808069 Children will discover the excit...


In [19]:
def retrieve_semantic_recommendations(
    query: str,
   top_k: int = 10,

)-> pd.DataFrame:
  recs= db.similarity_search(query,k=50)
  books_list=[]
  for i in range (0,len(recs)):
    books_list+= [int(recs[i].page_content.strip('"').split()[0])]

  return books[books["isbn13"].isin(books_list)].head(top_k)

In [22]:
retrieve_semantic_recommendations("A book to teach children about war")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
66,9780007162994,0007162995,If I Die in a Combat Zone,Tim O'Brien,"Vietnam War, 1961-1975",http://books.google.com/books/content?id=0qUtS...,Perhaps the best book to emerge from the Vietn...,2003.0,3.95,208.0,11.0,If I Die in a Combat Zone:,9780007162994 Perhaps the best book to emerge ...
404,9780064402453,0064402452,Racso and the Rats of NIMH,Jane Leslie Conly,Juvenile Fiction,http://books.google.com/books/content?id=MgoNv...,"‘Racso, a brash and boastful little rodent, is...",1988.0,3.76,288.0,3231.0,Racso and the Rats of NIMH:,"9780064402453 ‘Racso, a brash and boastful lit..."
524,9780099483472,0099483475,All Quiet on the Western Front,Erich Maria Remarque,"World War, 1914-1918",NaN,All Quiet on the Western Front is probably the...,2005.0,3.95,216.0,1018.0,All Quiet on the Western Front:,9780099483472 All Quiet on the Western Front i...
1006,9780195168952,019516895X,Battle Cry of Freedom,James M. McPherson,History,http://books.google.com/books/content?id=09FkZ...,Filled with fresh interpretations and informat...,2005.0,4.34,867.0,22318.0,Battle Cry of Freedom:The Civil War Era,9780195168952 Filled with fresh interpretation...
1199,9780312265052,0312265050,The Naked and the Dead,Norman Mailer,Fiction,http://books.google.com/books/content?id=c66GL...,Portrays the contrasting personalities and nos...,2000.0,3.94,721.0,20541.0,The Naked and the Dead:50th Anniversary Editio...,9780312265052 Portrays the contrasting persona...
2828,9780571207992,0571207995,The Wars,Timothy Findley,Fiction,http://books.google.com/books/content?id=AqnDQ...,"Robert Ross, a sensitive nineteen-year-old Can...",2001.0,3.87,218.0,6229.0,The Wars:,"9780571207992 Robert Ross, a sensitive ninetee..."
3139,9780684813219,0684813211,Achilles in Vietnam,Jonathan Shay,Psychology,http://books.google.com/books/content?id=6EEnD...,An original and groundbreaking book that exami...,1994.0,4.24,272.0,1057.0,Achilles in Vietnam:Combat Trauma and the Undo...,9780684813219 An original and groundbreaking b...
3154,9780684844077,0684844079,Soul of the Sword,Robert L. O'Connell,Technology & Engineering,http://books.google.com/books/content?id=eoEag...,A sweeping illustrated history of war and the ...,2002.0,4.08,400.0,36.0,Soul of the Sword:An Illustrated History of We...,9780684844077 A sweeping illustrated history o...
3180,9780688085872,0688085873,A Short History of World War II,James L. Stokesbury,History,http://books.google.com/books/content?id=uDBhl...,"Despite the numerous books on World War II, un...",1980.0,3.93,416.0,454.0,A Short History of World War II:,9780688085872 Despite the numerous books on Wo...
3331,9780743253970,0743253973,A Separate Peace,John Knowles,Fiction,http://books.google.com/books/content?id=MXodm...,An American classic and great bestseller for o...,2003.0,3.57,208.0,169719.0,A Separate Peace:,9780743253970 An American classic and great be...
